In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx


In [2]:
import pandas as pd

path = "/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx"

df = pd.read_excel(path)

df.head()

,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে ...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [3]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (6105, 4)

Columns:
['No.', 'Grievance Text', 'Category', 'Severity']


In [4]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
No.               0
Grievance Text    0
Category          0
Severity          0
dtype: int64


In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [6]:
category_count = df["Category"].value_counts()

print(category_count)

Category
Road & Transport    1061
Food                1036
Waste               1015
Healthcare          1015
Electricity         1007
Water                971
Name: count, dtype: int64


In [7]:
severity_count = df["Severity"].value_counts()

print(severity_count)

Severity
Medium    2464
Low       1988
High      1653
Name: count, dtype: int64


In [8]:
category_severity = pd.crosstab(
    df["Category"],
    df["Severity"]
)

category_severity

Severity,High,Low,Medium
Category,,,
Electricity,206,380,421
Food,168,393,475
Healthcare,342,294,379
Road & Transport,387,230,444
Waste,214,369,432
Water,336,322,313


In [9]:
df["text_length_words"] = df["Grievance Text"].astype(str).apply(
    lambda x: len(x.split())
)

print(df["text_length_words"].describe())

count    6105.000000
mean       21.408354
std        13.534100
min         2.000000
25%        12.000000
50%        18.000000
75%        26.000000
max       125.000000
Name: text_length_words, dtype: float64


In [10]:
print("Categories:")
print(df["Category"].unique())

print("\nSeverity:")
print(df["Severity"].unique())

Categories:
['Healthcare' 'Electricity' 'Water' 'Waste' 'Food' 'Road & Transport']

Severity:
['High' 'Low' 'Medium']


In [11]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [12]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [13]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [14]:
path = "/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx"

df = pd.read_excel(path)

df.head()

,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে ...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [15]:
data = df[["Grievance Text", "Category"]].copy()

data.columns = ["text", "label"]

label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(data["label"])

print(label_encoder.classes_)

num_labels = len(label_encoder.classes_)

print("Number of classes:", num_labels)

['Electricity' 'Food' 'Healthcare' 'Road & Transport' 'Waste' 'Water']
Number of classes: 6


In [16]:
train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data["label"]
)


valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (4273, 2)
Validation: (916, 2)
Test: (916, 2)


In [17]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [18]:
model_name = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [19]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )


train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/4273 [00:00<?, ? examples/s]

Map:   0%|          | 0/916 [00:00<?, ? examples/s]

Map:   0%|          | 0/916 [00:00<?, ? examples/s]

In [20]:
train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

valid_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

In [22]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision": precision_score(
            labels,
            predictions,
            average="macro"
        ),

        "recall": recall_score(
            labels,
            predictions,
            average="macro"
        ),

        "f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

In [23]:
training_args = TrainingArguments(
    output_dir="./banglabert_category",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=50,

    fp16=True,

    report_to="none"
)

In [24]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [25]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.260058,0.839247,0.924672,0.926650,0.923894,0.924105
2,0.464675,0.493222,0.936681,0.936398,0.936170,0.936055
3,0.312755,0.516359,0.936681,0.937409,0.935902,0.936130


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=804, training_loss=1.0907365025127704, metrics={'train_runtime': 245.9777, 'train_samples_per_second': 52.114, 'train_steps_per_second': 3.269, 'total_flos': 843235437860352.0, 'train_loss': 1.0907365025127704, 'epoch': 3.0})

In [26]:
result = trainer.evaluate(test_dataset)

result

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.5884544849395752,
 'eval_accuracy': 0.9301310043668122,
 'eval_precision': 0.932144973477472,
 'eval_recall': 0.9294871073253835,
 'eval_f1': 0.9301213589713648,
 'eval_runtime': 5.2425,
 'eval_samples_per_second': 174.724,
 'eval_steps_per_second': 11.063,
 'epoch': 3.0}

In [27]:
pred = trainer.predict(test_dataset)

y_pred = np.argmax(pred.predictions, axis=1)

y_true = test_df["label"].values


print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                  precision    recall  f1-score   support

     Electricity       0.96      0.95      0.95       151
            Food       0.94      0.95      0.94       156
      Healthcare       0.96      0.95      0.95       152
Road & Transport       0.90      0.94      0.92       159
           Waste       0.88      0.93      0.91       153
           Water       0.96      0.86      0.91       145

        accuracy                           0.93       916
       macro avg       0.93      0.93      0.93       916
    weighted avg       0.93      0.93      0.93       916



In [28]:
model_save_path = "./BanglaBERT_Category_Model"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!
